In [1]:
import os

In [2]:
# data_path=r"C:\Users\kumar\Downloads\archive (15)\garbage_classification_enhanced"
# for class_name in sorted(os.listdir(data_path)):
#     class_path=os.path.join(data_path,class_name)
#     if os.path.isdir(class_path):
#         Images=[
#             f for f in os.listdir(class_path)
#             if f.lower().endswith((".jpg",".jpeg",".png",".webp"))
#         ]
#         print(f"{class_name:15}:{len(Images)} images")

In [3]:
import tensorflow as tf

# Dataset location
dataset_path = r"C:\Users\kumar\Downloads\archive (15)\garbage_dataset_split"

# Image settings
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Training dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path + r"\train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

# Validation dataset
validation_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path + r"\validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Test dataset
test_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path + r"\test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("\nClasses:")
print(train_ds.class_names)

print("\nNumber of classes:", len(train_ds.class_names))

Found 11104 files belonging to 12 classes.
Found 2380 files belonging to 12 classes.
Found 2386 files belonging to 12 classes.

Classes:
['battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass']

Number of classes: 12


In [4]:
from tensorflow.keras import layers

# Data augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1)
])

# Normalization
normalization = layers.Rescaling(1.0 / 255)

In [5]:
# import matplotlib.pyplot as plt

# plt.figure(figsize=(10, 10))

# for images, labels in train_ds.take(1):

#     for i in range(9):
#         augmented_image = data_augmentation(images[i])

#         plt.subplot(3, 3, i + 1)
#         plt.imshow(augmented_image.numpy().astype("uint8"))
#         plt.title(train_ds.class_names[labels[i]])
#         plt.axis("off")

# plt.tight_layout()
# plt.show()

In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models

# =========================
# 1. SETTINGS
# =========================

dataset_path = r"C:\Users\kumar\Downloads\archive (15)\garbage_dataset_split"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 12


# =========================
# 2. LOAD DATA
# =========================

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path + r"\train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

validation_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path + r"\validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path + r"\test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# =========================
# 3. DATA AUGMENTATION
# =========================

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1)
])


# =========================
# 4. MOBILE NET V2
# =========================

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers
base_model.trainable = False


# =========================
# 5. BUILD MODEL
# =========================

inputs = layers.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

# MobileNetV2 preprocessing
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.2)(x)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

model = models.Model(inputs, outputs)


# =========================
# 6. COMPILE
# =========================

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# =========================
# 7. SHOW MODEL
# =========================

model.summary()

Found 11104 files belonging to 12 classes.
Found 2380 files belonging to 12 classes.
Found 2386 files belonging to 12 classes.


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 12)             │        15,372 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,273,356 (8.67 MB)

 Trainable params: 15,372 (60.05 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [7]:
base_model.trainable = False

In [8]:
import tensorflow as tf
import keras

In [ ]:
# model2=tf.keras.Sequential([
#     keras.layers.Conv2D(10,input_shape=(224,224,3),padding="same",kernel_size=(2,2),activation='relu'),
#     keras.layers.MaxPool2D(),
#     keras.layers.Conv2D(50,kernel_size=(2,2),activation='relu'),
#     keras.layers.MaxPool2D(),
#     keras.layers.Conv2D(100,kernel_size=(2,2),activation='relu'),
#     keras.layers.MaxPool2D(),
#     keras.layers.Flatten(),
#     keras.layers.Dense(500,activation="relu"),
#     keras.layers.Dense(100,activation="relu"),
#     keras.layers.Dense(50,activation="relu"),
#     keras.layers.Dense(12,activation="softmax")
#     ])
#  model2.compile(optimizer="adam",loss="sparse_categorical_crossentropy", metrics=['accuracy'])
#  model2.fit(train_ds,validation_data=validation_ds,epochs=5)

c:\Users\kumar\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [23]:
# model2.evaluate(test_ds)

In [51]:
history = model.fit(
       train_ds,
     validation_data=validation_ds,
     epochs=10
 )

Epoch 1/10


c:\Users\kumar\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


347/347 ━━━━━━━━━━━━━━━━━━━━ 147s 425ms/step - accuracy: 0.8835 - loss: 0.4002 - val_accuracy: 0.9130 - val_loss: 0.3072
Epoch 2/10
347/347 ━━━━━━━━━━━━━━━━━━━━ 148s 426ms/step - accuracy: 0.8904 - loss: 0.3644 - val_accuracy: 0.9164 - val_loss: 0.2878
Epoch 3/10
347/347 ━━━━━━━━━━━━━━━━━━━━ 148s 427ms/step - accuracy: 0.8952 - loss: 0.3448 - val_accuracy: 0.9189 - val_loss: 0.2694
Epoch 4/10
347/347 ━━━━━━━━━━━━━━━━━━━━ 148s 426ms/step - accuracy: 0.9010 - loss: 0.3313 - val_accuracy: 0.9239 - val_loss: 0.2571
Epoch 5/10
347/347 ━━━━━━━━━━━━━━━━━━━━ 146s 421ms/step - accuracy: 0.9028 - loss: 0.3121 - val_accuracy: 0.9265 - val_loss: 0.2469
Epoch 6/10
347/347 ━━━━━━━━━━━━━━━━━━━━ 146s 421ms/step - accuracy: 0.9105 - loss: 0.2960 - val_accuracy: 0.9273 - val_loss: 0.2365
Epoch 7/10
347/347 ━━━━━━━━━━━━━━━━━━━━ 158s 456ms/step - accuracy: 0.9093 - loss: 0.2878 - val_accuracy: 0.9298 - val_loss: 0.2314
Epoch 8/10
347/347 ━━━━━━━━━━━━━━━━━━━━ 178s 514ms/step - accuracy: 0.9107 - loss: 0.28

In [52]:
model.evaluate(test_ds)

75/75 ━━━━━━━━━━━━━━━━━━━━ 40s 538ms/step - accuracy: 0.9199 - loss: 0.2456


[0.24556970596313477, 0.9199497103691101]

In [31]:
# from sklearn.metrics import confusion_matrix,classification_report
# import numpy as np
# import numpy
# y_true=[]
# y_pred=[]
# for images,labels in test_ds:
#     predictions=model.predict(images,verbose=0)
#     predictions_class=np.argmax(predictions,axis=1)
#     y_true.extend(labels.numpy())
#     y_pred.extend(predictions_class)

# print("Classifications reports")
# print(classification_report(y_true,y_pred,target_names=train_ds.class_names))

# cm=confusion_matrix(y_true,y_pred)
# print(cm)



In [32]:
# import seaborn as sns
# import matplotlib.pyplot as plt
# sns.heatmap(cm,fmt="d",annot=True)
# plt.xlabel("predicated value")
# plt.ylabel("True")
# plt.show()

In [53]:
model.save("waste_classifier_10epochs.keras")

In [38]:
import numpy as np


In [43]:
for images, labels in train_ds.take(1):
    predictions = model.predict(images, verbose=0)

    for i in range(10):
        predicted_index = np.argmax(predictions[i])

        print(
            "Actual:",
            train_ds.class_names[labels[i]],
            "| Predicted:",
            train_ds.class_names[predicted_index],
            "| Confidence:",
            f"{predictions[i][predicted_index] * 100:.2f}%"
        )

Actual: clothes | Predicted: clothes | Confidence: 99.00%
Actual: brown-glass | Predicted: brown-glass | Confidence: 92.34%
Actual: clothes | Predicted: clothes | Confidence: 99.96%
Actual: battery | Predicted: battery | Confidence: 56.22%
Actual: clothes | Predicted: clothes | Confidence: 99.34%
Actual: plastic | Predicted: clothes | Confidence: 71.48%
Actual: shoes | Predicted: shoes | Confidence: 96.33%
Actual: green-glass | Predicted: green-glass | Confidence: 84.99%
Actual: metal | Predicted: metal | Confidence: 81.75%
Actual: biological | Predicted: biological | Confidence: 92.59%


In [45]:
image_path = r"C:\Users\kumar\Downloads\archive (15)\garbage_dataset_split\train\battery\battery99.jpg"

img = tf.keras.utils.load_img(
    image_path,
    target_size=(224, 224)
)

img_array = tf.keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0)

predictions = model.predict(img_array, verbose=0)[0]

predicted_index = np.argmax(predictions)

print("\nActual: battery")
print("Predicted:", train_ds.class_names[predicted_index])
print("Confidence:", predictions[predicted_index] * 100)


Actual: battery
Predicted: cardboard
Confidence: 35.39124
